## Quality of Care Indicators

Compute district-year quality-of-care indicators from DHIS2 outliers-imputed routine data.

Indicators:
- testing_rate = TEST / SUSP
- treatment_rate = MALTREAT / CONF
- case_fatality_rate = MALDTH / MALADM
- prop_adm_malaria = MALADM / ALLADM
- prop_malaria_deaths = MALDTH / ALLDTH
- non_malaria_all_cause_outpatients = ALLOUT (absolute)
- presumed_cases = PRES (absolute)

Stock-out indicators are not implemented yet (on hold, NMDR data pending).

In [ ]:
# Preliminaries
options(scipen=999)

ROOT_PATH <- "~/workspace"
CONFIG_PATH <- file.path(ROOT_PATH, "configuration")
CODE_PATH <- file.path(ROOT_PATH, "code")
DATA_PATH <- file.path(ROOT_PATH, "data")
OUTPUT_DATA_PATH <- file.path(DATA_PATH, "dhis2", "quality_of_care")
FIGURES_PATH <- file.path(ROOT_PATH, "pipelines", "snt_quality_of_care", "reporting", "outputs", "figures")

dir.create(OUTPUT_DATA_PATH, recursive = TRUE, showWarnings = FALSE)
dir.create(FIGURES_PATH, recursive = TRUE, showWarnings = FALSE)

source(file.path(CODE_PATH, "snt_utils.r"))
required_packages <- c("jsonlite", "data.table", "arrow", "sf", "ggplot2", "glue", "reticulate", "RColorBrewer", "writexl")
install_and_load(required_packages)

Sys.setenv(RETICULATE_PYTHON = "/opt/conda/bin/python")
openhexa <- reticulate::import("openhexa.sdk")

config_json <- jsonlite::fromJSON(file.path(CONFIG_PATH, "SNT_config.json"))
COUNTRY_CODE <- config_json$SNT_CONFIG$COUNTRY_CODE
DHIS2_FORMATTED_DATASET <- config_json$SNT_DATASET_IDENTIFIERS$DHIS2_DATASET_FORMATTED
OUTLIERS_DATASET <- config_json$SNT_DATASET_IDENTIFIERS$DHIS2_OUTLIERS_IMPUTATION

In [ ]:
# Fallback parameters for local/dev execution
if (!exists("outlier_imputation_method")) {
  outlier_imputation_method <- "mean"
}

allowed_methods <- c("mean", "median", "iqr", "trend", "mg-partial", "mg-complete")
if (!(outlier_imputation_method %in% allowed_methods)) {
  stop(glue::glue("Invalid outlier_imputation_method: {outlier_imputation_method}. Allowed: {paste(allowed_methods, collapse=', ')}"))
}

routine_filename <- glue::glue("{COUNTRY_CODE}_routine_outliers-{outlier_imputation_method}_imputed.parquet")
log_msg(glue::glue("Loading routine file from DHIS2 outliers dataset: {routine_filename}"))

routine <- get_latest_dataset_file_in_memory(OUTLIERS_DATASET, routine_filename)
shapes <- get_latest_dataset_file_in_memory(DHIS2_FORMATTED_DATASET, paste0(COUNTRY_CODE, "_shapes.geojson"))

setDT(routine)
required_cols <- c("ADM2_ID", "YEAR", "TEST", "SUSP", "MALTREAT", "CONF", "MALDTH", "MALADM", "ALLADM", "ALLDTH", "ALLOUT", "PRES")
missing_cols <- setdiff(required_cols, names(routine))
if (length(missing_cols) > 0) {
  stop(glue::glue("Missing required columns in routine data: {paste(missing_cols, collapse=', ')}"))
}

num_cols <- setdiff(required_cols, c("ADM2_ID", "YEAR"))
routine[, (num_cols) := lapply(.SD, function(x) as.numeric(x)), .SDcols = num_cols]
routine[, YEAR := as.integer(YEAR)]
routine[, ADM2_ID := as.character(ADM2_ID)]

qoc <- routine[, .(
  TEST = sum(TEST, na.rm = TRUE),
  SUSP = sum(SUSP, na.rm = TRUE),
  MALTREAT = sum(MALTREAT, na.rm = TRUE),
  CONF = sum(CONF, na.rm = TRUE),
  MALDTH = sum(MALDTH, na.rm = TRUE),
  MALADM = sum(MALADM, na.rm = TRUE),
  ALLADM = sum(ALLADM, na.rm = TRUE),
  ALLDTH = sum(ALLDTH, na.rm = TRUE),
  ALLOUT = sum(ALLOUT, na.rm = TRUE),
  PRES = sum(PRES, na.rm = TRUE)
), by = .(ADM2_ID, YEAR)]

qoc[, testing_rate := fifelse(SUSP > 0, TEST / SUSP, NA_real_)]
qoc[, treatment_rate := fifelse(CONF > 0, MALTREAT / CONF, NA_real_)]
qoc[, case_fatality_rate := fifelse(MALADM > 0, MALDTH / MALADM, NA_real_)]
qoc[, prop_adm_malaria := fifelse(ALLADM > 0, MALADM / ALLADM, NA_real_)]
qoc[, prop_malaria_deaths := fifelse(ALLDTH > 0, MALDTH / ALLDTH, NA_real_)]
qoc[, non_malaria_all_cause_outpatients := ALLOUT]
qoc[, presumed_cases := PRES]

shapes_dt <- as.data.table(sf::st_drop_geometry(shapes))
if ("ADM2_ID" %in% names(shapes_dt) && "ADM2_NAME" %in% names(shapes_dt)) {
  shapes_dt[, ADM2_ID := as.character(ADM2_ID)]
  qoc <- merge(qoc, unique(shapes_dt[, .(ADM2_ID, ADM2_NAME)]), by = "ADM2_ID", all.x = TRUE)
}

out_parquet <- file.path(OUTPUT_DATA_PATH, glue::glue("{COUNTRY_CODE}_quality_of_care_{outlier_imputation_method}.parquet"))
out_csv <- file.path(OUTPUT_DATA_PATH, glue::glue("{COUNTRY_CODE}_quality_of_care_{outlier_imputation_method}.csv"))
out_xlsx <- file.path(OUTPUT_DATA_PATH, glue::glue("{COUNTRY_CODE}_quality_of_care_{outlier_imputation_method}.xlsx"))

arrow::write_parquet(qoc, out_parquet)
data.table::fwrite(qoc, out_csv)
writexl::write_xlsx(list(quality_of_care = as.data.frame(qoc)), out_xlsx)

log_msg(glue::glue("Saved outputs: {out_parquet}, {out_csv}, {out_xlsx}"))

In [ ]:
# Yearly maps by ADM2
shapes$ADM2_ID <- as.character(shapes$ADM2_ID)
qoc$ADM2_ID <- as.character(qoc$ADM2_ID)

plot_yearly_map <- function(df, sf_shapes, value_col, title_prefix, filename_prefix, is_rate = TRUE) {
  years <- sort(unique(df$YEAR))
  for (yr in years) {
    df_y <- df[YEAR == yr]
    map_df <- merge(sf_shapes, df_y, by = "ADM2_ID", all.x = TRUE)

    p <- ggplot(map_df)

    if (is_rate) {
      map_df$cat <- cut(
        map_df[[value_col]],
        breaks = c(-Inf, 0, 0.2, 0.4, 0.6, 0.8, 1.0, Inf),
        labels = c("<0", "0-0.2", "0.2-0.4", "0.4-0.6", "0.6-0.8", "0.8-1.0", ">1.0"),
        include.lowest = TRUE
      )
      p <- p + geom_sf(aes(fill = cat), color = "grey60", size = 0.1) +
        scale_fill_brewer(palette = "YlOrRd", na.value = "white", drop = FALSE)
    } else {
      vals <- map_df[[value_col]]
      finite_vals <- vals[is.finite(vals)]
      if (length(finite_vals) > 4) {
        br <- unique(as.numeric(quantile(finite_vals, probs = seq(0, 1, 0.2), na.rm = TRUE)))
        if (length(br) < 2) {
          map_df$cat <- as.factor("all")
        } else {
          map_df$cat <- cut(vals, breaks = br, include.lowest = TRUE)
        }
      } else {
        map_df$cat <- as.factor(vals)
      }
      p <- p + geom_sf(aes(fill = cat), color = "grey60", size = 0.1) +
        scale_fill_brewer(palette = "Blues", na.value = "white", drop = FALSE)
    }

    p <- p +
      theme_void() +
      labs(
        title = paste0(title_prefix, " - ", yr),
        fill = value_col,
        caption = "Source: SNT DHIS2 outliers-imputed routine data"
      ) +
      theme(
        legend.position = "bottom",
        plot.title = element_text(face = "bold", size = 12)
      )

    out_png <- file.path(FIGURES_PATH, glue::glue("{filename_prefix}_{yr}.png"))
    ggsave(out_png, plot = p, width = 9, height = 7, dpi = 300, bg = "white")
  }
}

plot_yearly_map(qoc, shapes, "testing_rate", "Testing rate (TEST / SUSP)", "testing_rate", TRUE)
plot_yearly_map(qoc, shapes, "treatment_rate", "Treatment rate (MALTREAT / CONF)", "treatment_rate", TRUE)
plot_yearly_map(qoc, shapes, "case_fatality_rate", "In-hospital case fatality rate (MALDTH / MALADM)", "case_fatality_rate", TRUE)
plot_yearly_map(qoc, shapes, "prop_adm_malaria", "Proportion admitted for malaria (MALADM / ALLADM)", "prop_adm_malaria", TRUE)
plot_yearly_map(qoc, shapes, "prop_malaria_deaths", "Proportion of malaria deaths (MALDTH / ALLDTH)", "prop_malaria_deaths", TRUE)
plot_yearly_map(qoc, shapes, "non_malaria_all_cause_outpatients", "Non-malaria all-cause outpatients (ALLOUT)", "allout", FALSE)
plot_yearly_map(qoc, shapes, "presumed_cases", "Presumed cases (PRES)", "presumed_cases", FALSE)

log_msg(glue::glue("Saved yearly maps in: {FIGURES_PATH}"))